# Newspaper Article Retrieval — 2024 US Presidential Election

Retrieves election-related news articles from politically diverse outlets for **4 months before** the election: **Jul 5 – Nov 4, 2024**.

| Source | Leaning | Method |
|---|---|---|
| The Guardian | Democratic | Guardian Open Platform API (free) |
| New York Times | Democratic | NYT Article Search API (free) |
| NBC News | Democratic | GDELT DOC 2.0 API (no key needed) |
| Fox News | Republican | GDELT DOC 2.0 API (no key needed) |
| Breitbart | Republican | GDELT DOC 2.0 API (no key needed) |
| NY Post | Republican | GDELT DOC 2.0 API (no key needed) |
| Daily Caller | Republican | GDELT DOC 2.0 API (no key needed) |

**API keys needed:**
- Guardian: https://open-platform.theguardian.com/access/
- NYT: https://developer.nytimes.com/get-started
- GDELT: no key required

In [1]:
import requests
import pandas as pd
import time
import os

# ── API Keys ─────────────────────────────────────────────────────────────────
GUARDIAN_API_KEY = "47346645-10a7-4288-aa66-7a7191f8f9eb"
NYT_API_KEY      = "QvQ4PUkpkCtnwmwkO1bLJNgfbkmhCK2NRh5PD4ypHZeAGpNr"

# ── Date range: 3 months before the election (Nov 5, 2024) ───────────────────
START_DATE = "2024-07-05"
END_DATE   = "2024-11-04"

# ── Search query ──────────────────────────────────────────────────────────────
QUERY = "Trump OR Harris presidential election 2024"

# ── Final columns ─────────────────────────────────────────────────────────────
COLS = ["source", "leaning", "date", "title", "url"]

os.makedirs("data", exist_ok=True)
print(f"Date range: {START_DATE} to {END_DATE}")

Date range: 2024-07-05 to 2024-11-04


## 1. The Guardian API (Democratic-leaning)

In [2]:
def fetch_guardian(query, from_date, to_date, api_key):
    """Fetch all available pages (no cap)."""
    articles = []
    page = 1
    while True:
        params = {
            "q":           query,
            "from-date":   from_date,
            "to-date":     to_date,
            "api-key":     api_key,
            "page-size":   200,
            "page":        page,
            "show-fields": "headline",
            "order-by":    "oldest",
        }
        r = requests.get("https://content.guardianapis.com/search", params=params, timeout=30)
        r.raise_for_status()
        response    = r.json()["response"]
        results     = response.get("results", [])
        total_pages = response.get("pages", 1)

        if not results:
            break

        for item in results:
            articles.append({
                "source":  "The Guardian",
                "leaning": "Democratic",
                "date":    item["webPublicationDate"][:10],
                "title":   item.get("fields", {}).get("headline", item.get("webTitle", "")),
                "url":     item.get("webUrl", ""),
            })

        print(f"  page {page}/{total_pages} — {len(results)} articles (total so far: {len(articles)})")
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.2)

    return pd.DataFrame(articles)

print("Fetching Guardian articles (all pages)...")
df_guardian = fetch_guardian(QUERY, START_DATE, END_DATE, GUARDIAN_API_KEY)
print(f"Total: {len(df_guardian)} articles")
df_guardian.head(3)

Fetching Guardian articles (all pages)...
  page 1/56 — 200 articles (total so far: 200)
  page 2/56 — 200 articles (total so far: 400)
  page 3/56 — 200 articles (total so far: 600)
  page 4/56 — 200 articles (total so far: 800)
  page 5/56 — 200 articles (total so far: 1000)
  page 6/56 — 200 articles (total so far: 1200)
  page 7/56 — 200 articles (total so far: 1400)
  page 8/56 — 200 articles (total so far: 1600)
  page 9/56 — 200 articles (total so far: 1800)
  page 10/56 — 200 articles (total so far: 2000)
  page 11/56 — 200 articles (total so far: 2200)
  page 12/56 — 200 articles (total so far: 2400)
  page 13/56 — 200 articles (total so far: 2600)
  page 14/56 — 200 articles (total so far: 2800)
  page 15/56 — 200 articles (total so far: 3000)
  page 16/56 — 200 articles (total so far: 3200)
  page 17/56 — 200 articles (total so far: 3400)
  page 18/56 — 200 articles (total so far: 3600)
  page 19/56 — 200 articles (total so far: 3800)
  page 20/56 — 200 articles (total so fa

,source,leaning,date,title,url
0,The Guardian,Democratic,2024-07-05,Ukraine war briefing: Ukrainian army confirms ...,https://www.theguardian.com/world/article/2024...
1,The Guardian,Democratic,2024-07-05,Nigel Farage hails Reform UK’s ‘almost unbelie...,https://www.theguardian.com/politics/article/2...
2,The Guardian,Democratic,2024-07-05,"The seconds ticked by to 10pm, when the people...",https://www.theguardian.com/politics/article/2...


## 2. New York Times Article Search API (Democratic-leaning)

In [3]:
def fetch_nyt(query, begin_date, end_date, api_key, max_pages=100):
    """10 articles per page, max 100 pages = up to 1000 articles."""
    base_url  = "https://api.nytimes.com/svc/search/v2/articlesearch.json"
    articles  = []
    begin_fmt = begin_date.replace("-", "")
    end_fmt   = end_date.replace("-", "")

    for page in range(max_pages):
        params = {
            "q":          query,
            "begin_date": begin_fmt,
            "end_date":   end_fmt,
            "api-key":    api_key,
            "page":       page,
            "sort":       "oldest",
            "fl":         "headline,pub_date,web_url",
        }
        for attempt in range(3):
            r = requests.get(base_url, params=params, timeout=30)
            if r.status_code in (401, 429):
                wait = 15 * (attempt + 1)
                print(f"  Rate limited (attempt {attempt+1}), waiting {wait}s...")
                time.sleep(wait)
                continue
            r.raise_for_status()
            break
        else:
            print(f"  Skipping page {page+1} after 3 failed attempts.")
            continue

        docs = r.json()["response"]["docs"]
        if not docs:
            print(f"  No more results at page {page+1}, stopping.")
            break

        for doc in docs:
            articles.append({
                "source":  "New York Times",
                "leaning": "Democratic",
                "date":    doc.get("pub_date", "")[:10],
                "title":   doc.get("headline", {}).get("main", ""),
                "url":     doc.get("web_url", ""),
            })

        print(f"  page {page+1} — {len(docs)} articles (total so far: {len(articles)})")
        time.sleep(12)  # 12s gap → safely under 10 req/min

    return pd.DataFrame(articles)

print("Fetching NYT articles (up to 1000)...")
df_nyt = fetch_nyt(QUERY, START_DATE, END_DATE, NYT_API_KEY, max_pages=100)
print(f"Total: {len(df_nyt)} articles")
df_nyt.head(3)

Fetching NYT articles (up to 1000)...
  page 1 — 10 articles (total so far: 10)
  page 2 — 10 articles (total so far: 20)
  page 3 — 10 articles (total so far: 30)
  page 4 — 10 articles (total so far: 40)
  page 5 — 10 articles (total so far: 50)
  page 6 — 10 articles (total so far: 60)
  page 7 — 10 articles (total so far: 70)
  page 8 — 10 articles (total so far: 80)
  page 9 — 10 articles (total so far: 90)
  page 10 — 10 articles (total so far: 100)
  page 11 — 10 articles (total so far: 110)
  page 12 — 10 articles (total so far: 120)
  page 13 — 10 articles (total so far: 130)
  page 14 — 10 articles (total so far: 140)
  page 15 — 10 articles (total so far: 150)
  page 16 — 10 articles (total so far: 160)
  page 17 — 10 articles (total so far: 170)
  page 18 — 10 articles (total so far: 180)
  page 19 — 10 articles (total so far: 190)
  page 20 — 10 articles (total so far: 200)
  page 21 — 10 articles (total so far: 210)
  page 22 — 10 articles (total so far: 220)
  page 23 — 

,source,leaning,date,title,url
0,New York Times,Democratic,2024-07-05,Is Kamala Harris Underrated?,https://www.nytimes.com/2024/07/05/opinion/ezr...
1,New York Times,Democratic,2024-07-05,These Voters Supported Biden in 2020. Now They...,https://www.nytimes.com/2024/07/05/us/election...
2,New York Times,Democratic,2024-07-05,"Biden Campaign, Sticking to Its Playbook, Will...",https://www.nytimes.com/2024/07/05/us/politics...


## 3. GDELT DOC 2.0 — Fox News, Breitbart, NY Post (Republican) + NBC News (Democratic)

GDELT monitors global news 24/7 across thousands of outlets. No API key required.

In [4]:
def fetch_gdelt_domain(keyword, domain, start_dt, end_dt, max_records=250):
    params = {
        "query":         f"{keyword} domain:{domain}",
        "mode":          "artlist",
        "maxrecords":    max_records,
        "startdatetime": start_dt,
        "enddatetime":   end_dt,
        "format":        "json",
        "sort":          "DateAsc",
    }
    for attempt in range(5):
        try:
            r = requests.get("https://api.gdeltproject.org/api/v2/doc/doc",
                             params=params, timeout=60)
            if r.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"    Rate limited, waiting {wait}s (attempt {attempt+1}/5)...")
                time.sleep(wait)
                continue
            r.raise_for_status()
            if not r.text.strip():
                return []
            return r.json().get("articles", [])
        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f"    Error: {e} — retrying in {wait}s...")
            time.sleep(wait)
    print(f"    Failed after 5 attempts for {domain}, skipping.")
    return []


def fetch_gdelt_all(keyword, domain, source_name, leaning, start_date, end_date):
    """
    Split the date range into weekly chunks to bypass the 250-article cap.
    Each chunk gets up to 250 articles → ~13 weeks × 250 = up to 3250 per source.
    """
    chunks = pd.date_range(start=start_date, end=end_date, freq="W")
    boundaries = [pd.Timestamp(start_date)] + list(chunks) + [pd.Timestamp(end_date)]
    boundaries = sorted(set(boundaries))

    rows = []
    for i in range(len(boundaries) - 1):
        chunk_start = boundaries[i].strftime("%Y%m%d") + "000000"
        chunk_end   = boundaries[i+1].strftime("%Y%m%d") + "235959"
        results = fetch_gdelt_domain(keyword, domain, chunk_start, chunk_end)
        for item in results:
            raw_date = item.get("seendate", "")[:8]
            try:
                date_fmt = pd.to_datetime(raw_date, format="%Y%m%d").strftime("%Y-%m-%d")
            except Exception:
                date_fmt = ""
            rows.append({
                "source":  source_name,
                "leaning": leaning,
                "date":    date_fmt,
                "title":   item.get("title", ""),
                "url":     item.get("url", ""),
            })
        time.sleep(5)

    return rows


SOURCES = {
    "nbcnews.com":     ("NBC News",     "Democratic"),
    "foxnews.com":     ("Fox News",     "Republican"),
    "breitbart.com":   ("Breitbart",    "Republican"),
    "nypost.com":      ("NY Post",      "Republican"),
    "dailycaller.com": ("Daily Caller", "Republican"),
}

keyword  = "Trump Harris election"
gdelt_rows = []

print("Fetching GDELT articles (weekly chunks)...")
for domain, (source_name, leaning) in SOURCES.items():
    before = len(gdelt_rows)
    gdelt_rows += fetch_gdelt_all(keyword, domain, source_name, leaning, START_DATE, END_DATE)
    print(f"  [{source_name}]: {len(gdelt_rows) - before} articles")
    time.sleep(10)

df_gdelt = pd.DataFrame(gdelt_rows).drop_duplicates(subset="url").reset_index(drop=True)
print(f"\nGDELT total: {len(df_gdelt)} articles")
df_gdelt.head(3)

Fetching GDELT articles (weekly chunks)...
    Rate limited, waiting 30s (attempt 1/5)...
    Rate limited, waiting 30s (attempt 1/5)...
    Rate limited, waiting 60s (attempt 2/5)...
    Rate limited, waiting 30s (attempt 1/5)...
    Error: HTTPSConnectionPool(host='api.gdeltproject.org', port=443): Max retries exceeded with url: /api/v2/doc/doc?query=Trump+Harris+election+domain%3Anbcnews.com&mode=artlist&maxrecords=250&startdatetime=20240825000000&enddatetime=20240901235959&format=json&sort=DateAsc (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.gdeltproject.org', port=443) at 0x11e5cc950>, 'Connection to api.gdeltproject.org timed out. (connect timeout=60)')) — retrying in 30s...
    Rate limited, waiting 90s (attempt 3/5)...
    Rate limited, waiting 30s (attempt 1/5)...
    Rate limited, waiting 60s (attempt 2/5)...
    Rate limited, waiting 90s (attempt 3/5)...
    Rate limited, waiting 30s (attempt 1/5)...
    Rate limited, waiting 30s (attempt 1/5)...
  [NBC News]: 5

,source,leaning,date,title,url
0,NBC News,Democratic,2024-07-05,Biden doubles - down at Wisconsin rally : Im ...,https://www.nbcnews.com/politics/2024-election...
1,NBC News,Democratic,2024-07-05,Trump desire to win Black voters may be at odd...,https://www.nbcnews.com/news/nbcblk/trump-stop...
2,NBC News,Democratic,2024-07-05,"Democratic donors divided on what comes next ,...",https://www.nbcnews.com/politics/2024-election...


## 4. Combine All Sources & Save

In [5]:
df_all = pd.concat([df_guardian, df_nyt, df_gdelt], ignore_index=True)[COLS]

# Normalise dates and filter range
df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")
mask = (df_all["date"] >= START_DATE) & (df_all["date"] <= END_DATE)
df_all = df_all[mask].drop_duplicates(subset="url").sort_values("date").reset_index(drop=True)

print(f"Before keyword filter : {len(df_all)} articles")

# ── Keyword filter: keep only US election-relevant articles ──────────────────
US_KEYWORDS = [
    # Candidates & key figures
    "trump", "harris", "kamala", "donald", "melania",
    "biden", "obama", "pence", "walz", "vance",
    "hillary", "clinton",

    # Parties & movements
    "republican", "democrat", "democratic", "republic", "gop", "maga",
    "far-right", "far right", "liberal", "conservative",
    "progressive", "populist",

    # Election mechanics
    "presidential", "election", "ballot", "vote", "voter", "voting",
    "poll", "polls", "polling", "electorate", "electoral college",
    "swing state", "battleground", "toss-up",
    "primary", "caucus", "convention", "nomination", "nominee",
    "early voting", "mail-in", "absentee", "voter suppression",
    "recount", "results", "election day", "november 5",

    # Institutions & places
    "white house", "oval office", "congress", "senate", "house of representatives",
    "supreme court", "federal", "washington d.c.", "pentagon",

    # Campaign & politics
    "campaign", "debate", "rally", "fundraising", "super pac",
    "approval rating", "swing voters", "independents",
    "running mate", "vice president", "inauguration",
    "concede", "concession", "transition",

    # Key issues
    "abortion", "immigration", "border", "economy", "inflation",
    "gun control", "healthcare", "climate", "foreign policy",
    "ukraine", "israel", "gaza", "nato",
    "tariff", "trade", "jobs", "taxes", "deficit",
    "crime", "fentanyl", "opioid",

    # Media & polling
    "fox news", "cnn", "msnbc", "nbc news", "new york times",
    "gallup", "reuters", "associated press",
    "misinformation", "disinformation", "fact-check",

    # Prediction markets & odds
    "polymarket", "prediction market", "odds", "forecast",
]

pattern = "|".join(US_KEYWORDS)
mask_kw = df_all["title"].str.lower().str.contains(pattern, na=False)
df_all = df_all[mask_kw].reset_index(drop=True)

print(f"After keyword filter  : {len(df_all)} articles")
print(f"Removed               : {(~mask_kw).sum()} irrelevant articles\n")
print("=== Combined dataset ===")
print(df_all.groupby(["source", "leaning"]).size().to_string())
print(f"\nDate range: {df_all['date'].min().date()} → {df_all['date'].max().date()}")
df_all.head()

Before keyword filter : 15438 articles
After keyword filter  : 7684 articles
Removed               : 7754 irrelevant articles

=== Combined dataset ===
source          leaning   
Breitbart       Republican     653
Daily Caller    Republican     378
Fox News        Republican     958
NBC News        Democratic     435
NY Post         Republican     644
New York Times  Democratic     823
The Guardian    Democratic    3793

Date range: 2024-07-05 → 2024-11-04


,source,leaning,date,title,url
0,The Guardian,Democratic,2024-07-05,Ukraine war briefing: Ukrainian army confirms ...,https://www.theguardian.com/world/article/2024...
1,The Guardian,Democratic,2024-07-05,"Labour bounceback, Tory collapse: five key tak...",https://www.theguardian.com/politics/article/2...
2,The Guardian,Democratic,2024-07-05,French PM urges calm after assaults in run-up ...,https://www.theguardian.com/world/article/2024...
3,The Guardian,Democratic,2024-07-05,‘Goodwill on all sides’: transfer of UK power ...,https://www.theguardian.com/politics/article/2...
4,The Guardian,Democratic,2024-07-05,Labour convinced voters it was the safest choi...,https://www.theguardian.com/politics/article/2...


In [ ]:
df_all.to_csv("Data/1_Bronze/Newspaper/data.csv", index=False)
df_all[df_all["leaning"] == "Democratic"].to_csv("Data/1_Bronze/Newspaper/data.csv", index=False)
df_all[df_all["leaning"] == "Republican"].to_csv("Data/1_Bronze/Newspaper/data.csv", index=False)

print("Saved!")

Saved:
  data/newspaper_articles.csv   (all sources)
  data/newspaper_democratic.csv (Guardian, NYT, NBC News)
  data/newspaper_republican.csv (Fox News, Breitbart, NY Post, Daily Caller)
